# Polymer assembly: a PEG-like chain

Assemble a small polymer from building blocks and parametrise it.

* **Building blocks**: ethanediol (`ethandial`, 2 reactive ends) and a PEI-type cap (`pei`).
* **Assembly**: three ethanediol units in a row (`A`), capped with `pei` (`B`) - 68 atoms.
* **Output**: GROMACS topology whose building-block terms are transferred GAFF parameters
  and whose junction terms are derived by PolyLego.

Runtime: about 3 minutes.

## Setup
Checks that PolyLego, GROMACS and AmberTools are available and pins the thread counts.

In [1]:
import os, sys, json
sys.path.insert(0, os.path.abspath(os.path.join("..", "common")))
from polylego_examples import *

# Pin threads for every step launched from this notebook (see examples/README.md).
os.environ.setdefault("OMP_NUM_THREADS", "8")
for k in ("OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ[k] = "1"
os.environ.setdefault("PYTHONHASHSEED", "0")

GMX = check_env()
os.environ["POLYLEGO_GMX"] = GMX
# GPU for the GROMACS runs: "auto" (use a compatible GPU if there is one, else CPU),
# "none" (CPU only), or a GPU id such as "0".
GPU = os.environ.get("POLYLEGO_GPU", "auto")
os.makedirs("output", exist_ok=True)

PolyLego   1.4.0  ($CONDA_PREFIX/lib/python3.10/site-packages/PolyLego)
Python     3.10.13
GROMACS    $CONDA_PREFIX/bin/gmx
AmberTools $CONDA_PREFIX/bin/antechamber
OMP_NUM_THREADS        8
OPENBLAS_NUM_THREADS   1
MKL_NUM_THREADS        1


## 1. Inputs
The assembly definition lists the building blocks with their reactive groups (SMARTS, bonding atom, atoms removed on bonding) and the connection rules.

In [2]:
print(open("input/peg.txt").read())

## [ BUILDING BLOCK ]
name	type	head_smart	head_bonders	head_deletions	end_smart	end_bonder	end_deletor
ethandial	2	[C][O][H]	0	1,2	[C][O][H]	1	2
pei	3	[O]=[C]-[O]-[H]	1	2,3

## [ CONNECTION ]
name	donate_mol	accept_mol	repeat	reaction_times
A	ethandial	ethandial	3	1
B	A	pei	1	1



In [3]:
show_grid(["input/ethandial.sdf", "input/pei.sdf"], ["ethandial", "pei"], cols=2)

[3D view - run the notebook in Jupyter to display it]


## 2. Assemble the polymer
`PolyLego-BuildPolymer` builds the molecular graph of assembly `B`. The coordinates in `B.sdf` are a graph layout (every bond 1.5 A), not a 3D conformation - that comes in step 3.

In [4]:
run(["PolyLego-BuildPolymer", "-i", "input", "-o", "output",
     "-t", "input/peg.txt", "-l", "B"], log="output/build.log")
show_mol("output/B.sdf")

$ PolyLego-BuildPolymer -i input -o output -t input/peg.txt -l B


  PolyLego 1.4.0 | BuildPolymer
  assembling B from input/peg.txt
  Finished in 0 s
    output/B.sdf
    output/B_graph.gml
  -> exit 0, 1 s


[3D view - run the notebook in Jupyter to display it]


## 3. Assign force-field parameters
`PolyLego-AssignForceField` parametrises every building block, derives the junction terms from capped fragments, merges them into `assembly.top`, and relaxes the structure with a short vacuum EM/NVT.

In [5]:
run(["PolyLego-AssignForceField", "-l", "input", "-t", "input/peg.txt",
     "-o", "output", "-n", "B", "--gpu", GPU], log="output/assign.log")

$ PolyLego-AssignForceField -l input -t input/peg.txt -o output -n B --gpu none


  PolyLego 1.4.0 | AssignForceField
  force field for B (polymer)
    building block pei: GAFF-2.11 + AM1-BCC charges
    building block ethandial: GAFF-2.11 + AM1-BCC charges
    junction parameters from capped fragments
    assembly topology
    vacuum energy minimisation + short NVT (GROMACS)
  Finished in 2 min 49 s
    output/assembly.top
    output/assembly.gro
    output/assembly.pdb
  -> exit 0, 171 s


In [6]:
show_mol("output/assembly.mol2")

[3D view - run the notebook in Jupyter to display it]


## 4. Check the topology

In [7]:
import parmed
def topology_summary(top, gro):
    s = parmed.load_file(top, xyz=gro)
    n_imp = sum(d.improper for d in s.dihedrals) + len(s.impropers)
    print(f"atoms            {len(s.atoms)}")
    print(f"bonds            {len(s.bonds)}")
    print(f"angles           {len(s.angles)}")
    print(f"proper dihedrals {sum(not d.improper for d in s.dihedrals)}")
    print(f"impropers        {n_imp}")
    print(f"net charge       {sum(a.charge for a in s.atoms):+.4f} e")
    return s

_ = topology_summary("output/assembly.top", "output/assembly.gro")

  warnings.warn('%i 1-4 pairs were missing from the [ pairs ] '



atoms            68
bonds            67
angles           122
proper dihedrals 147
impropers        2
net charge       -0.0000 e


`output/assembly.top` + `output/assembly.gro` are ready for `gmx grompp`.